# Clasificación supervisada de la popularidad musical mediante XGBoost

## Objetivo

El objetivo de este notebook es desarrollar y evaluar un modelo de clasificación supervisada capaz de predecir la categoría de popularidad de una canción a partir de sus características musicales.

Para ello, se utilizará el algoritmo XGBoost (Extreme Gradient Boosting), un método de ensamblaje basado en árboles de decisión que construye modelos de manera secuencial. Cada nuevo árbol busca corregir los errores cometidos por los modelos anteriores mediante la optimización de una función de pérdida.

El análisis contempla las siguientes etapas:

- Carga de los datasets de entrenamiento y prueba.
- Separación de las variables predictoras y la variable objetivo.
- Búsqueda de hiperparámetros mediante validación cruzada estratificada.
- Entrenamiento del modelo XGBoost con los mejores hiperparámetros encontrados.
- Evaluación del rendimiento durante la búsqueda de hiperparámetros.
- Evaluación final sobre el conjunto de prueba.
- Análisis de las matrices de confusión.
- Identificación de las características con mayor importancia para el modelo.

La variable objetivo será `popularity_category`, que representa la categoría de popularidad asignada a cada canción.

**Consideración:** se asume que los datasets ya se encuentran limpios y codificados. Por tanto, no se aplican transformaciones adicionales ni codificación de la variable objetivo dentro de este notebook.

In [ ]:
import pandas as pd
import os
import sys

import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))

---

## Carga de los datasets de entrenamiento y prueba

Se cargan dos datasets previamente preprocesados:

- **Conjunto de entrenamiento:** se utiliza para buscar los mejores hiperparámetros y ajustar el modelo.
- **Conjunto de prueba:** se reserva para evaluar el rendimiento final del modelo sobre observaciones que no participan en la búsqueda ni en el entrenamiento.

Ambos conjuntos deben mantener la misma estructura de variables predictoras, incluyendo el mismo orden y las mismas transformaciones aplicadas previamente.

Finalmente, se muestran las primeras filas de cada dataset para verificar su carga y observar la estructura de los datos.

In [ ]:
data_train = pd.read_csv(
    "../../../dataset/Spotify_train_preprocesado_clasificacion.csv"
)

data_test = pd.read_csv(
    "../../../dataset/Spotify_test_preprocesado_clasificacion.csv"
)

df_train = data_train.copy()
df_test = data_test.copy()

print("Dataset de entrenamiento:")
display(df_train.head())

print("\nDataset de prueba:")
display(df_test.head())

---

## Separación de variables predictoras y variable objetivo

Se separan las variables utilizadas para realizar las predicciones de la variable que se desea estimar.

- **X_train:** características musicales del conjunto de entrenamiento.
- **y_train:** categorías de popularidad correspondientes al conjunto de entrenamiento.
- **X_test:** características musicales del conjunto de prueba.
- **y_test:** categorías reales de popularidad del conjunto de prueba.

La variable `popularity_category` se excluye de las variables predictoras para evitar que el modelo utilice directamente la respuesta que debe aprender a predecir.

Se utilizan directamente las categorías codificadas presentes en los datasets, sin aplicar transformaciones adicionales.

**Requisito de XGBoost:** en problemas multiclase, las etiquetas deben estar representadas por números enteros consecutivos desde cero (0, 1, 2, ...). Se asume que esta condición ya se cumple en los datos proporcionados.

In [ ]:
# Separación de variables predictoras y objetivo
X_train = df_train.drop(
    columns=["popularity_category"]
)

y_train = df_train["popularity_category"]

X_test = df_test.drop(
    columns=["popularity_category"]
)

y_test = df_test["popularity_category"]

---

## Búsqueda de hiperparámetros mediante RandomizedSearchCV

Antes de entrenar el modelo definitivo, se realiza una búsqueda de hiperparámetros para identificar una configuración que permita mejorar el rendimiento de XGBoost.

Se utiliza `RandomizedSearchCV`, que selecciona aleatoriamente combinaciones de valores dentro de un espacio de búsqueda definido. A diferencia de una búsqueda exhaustiva, este procedimiento permite explorar distintas configuraciones sin evaluar todas las combinaciones posibles.

Se consideran los siguientes hiperparámetros:

- `n_estimators`: cantidad de árboles que se construyen secuencialmente.
- `learning_rate`: controla la contribución de cada árbol nuevo al modelo.
- `max_depth`: profundidad máxima de los árboles, relacionada con su complejidad.
- `min_child_weight`: controla la cantidad mínima de información requerida en un nodo hijo para permitir una división.
- `subsample`: proporción de observaciones utilizadas para construir cada árbol.
- `colsample_bytree`: proporción de variables predictoras utilizadas por árbol.
- `gamma`: reducción mínima de la función de pérdida requerida para realizar una división.
- `reg_alpha`: regularización L1, que puede ayudar a controlar la complejidad del modelo.
- `reg_lambda`: regularización L2, que ayuda a evitar modelos excesivamente complejos.

La búsqueda utiliza validación cruzada estratificada de cinco particiones. En cada iteración, el modelo se entrena con cuatro particiones y se evalúa con la restante, procurando mantener la distribución de las categorías.

Se utiliza **F1-Macro** como métrica principal para seleccionar la mejor configuración, ya que asigna la misma importancia a todas las categorías de popularidad, independientemente de su frecuencia.

También se registran Accuracy, Balanced Accuracy, Precision Weighted, Recall Weighted y F1 Weighted para analizar el comportamiento de las configuraciones evaluadas.

El parámetro `refit=True` permite que, una vez identificada la mejor combinación, el estimador se vuelva a entrenar automáticamente utilizando la totalidad del conjunto de entrenamiento.

**Consideración metodológica:** el conjunto de prueba no participa en la búsqueda de hiperparámetros. De esta manera, se reserva para la evaluación final de la capacidad de generalización del modelo.

In [ ]:
# Configuración de la validación cruzada
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Modelo base de XGBoost
xgb_base = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=1
)

# Espacio de búsqueda de hiperparámetros
param_distributions = {
    "n_estimators": [200, 300, 400, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 3, 5]
}

# Métricas utilizadas durante la búsqueda
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",
    "f1_macro": "f1_macro"
}

# Búsqueda aleatoria de hiperparámetros
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=30,
    scoring=scoring,
    refit="f1_macro",
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

# Ejecución de la búsqueda
random_search.fit(X_train, y_train)

---

## Resultados de la búsqueda de hiperparámetros

Una vez finalizada la búsqueda, se identifica la combinación de hiperparámetros que obtuvo el mejor resultado promedio de F1-Macro durante la validación cruzada.

Se presentan los mejores parámetros encontrados y las métricas obtenidas por dicha configuración.

Además, se muestra una tabla con las diez mejores combinaciones evaluadas, ordenadas según su F1-Macro promedio.

La desviación estándar permite observar la variabilidad del rendimiento entre las cinco particiones. Un valor menor indica resultados más consistentes entre las particiones evaluadas.

In [ ]:
print("Mejores hiperparámetros encontrados:")

for parametro, valor in random_search.best_params_.items():
    print(f"{parametro}: {valor}")

print(
    f"\nMejor F1-Macro promedio: "
    f"{random_search.best_score_:.4f}"
)

# Obtener resultados de todas las configuraciones evaluadas
resultados_busqueda = pd.DataFrame(random_search.cv_results_)

columnas_resultados = [
    "rank_test_f1_macro",
    "mean_test_accuracy",
    "std_test_accuracy",
    "mean_test_balanced_accuracy",
    "mean_test_f1_weighted",
    "mean_test_f1_macro",
    "std_test_f1_macro",
    "params"
]

resultados_ordenados = resultados_busqueda[
    columnas_resultados
].sort_values(
    by="rank_test_f1_macro"
)

print("\nTop 10 configuraciones evaluadas:")

display(resultados_ordenados.head(10))

---

## Visualización de las métricas de la mejor configuración

Se representan las métricas promedio obtenidas por la mejor configuración de hiperparámetros durante la validación cruzada.

Esta visualización permite observar el rendimiento del modelo desde distintas perspectivas, considerando tanto el desempeño general como el comportamiento frente a las diferentes categorías.

In [ ]:
mejor_indice = random_search.best_index_

metricas_xgb = [
    "accuracy",
    "balanced_accuracy",
    "f1_weighted",
    "f1_macro"
]

valores_metricas = [
    random_search.cv_results_[f"mean_test_{metrica}"][mejor_indice]
    for metrica in metricas_xgb
]

df_metricas_xgb = pd.DataFrame({
    "Métrica": metricas_xgb,
    "Valor": valores_metricas
})

fig, ax = plt.subplots(figsize=(11, 6))

sns.barplot(
    data=df_metricas_xgb,
    x="Métrica",
    y="Valor",
    color="steelblue",
    ax=ax
)

ax.set_title(
    "XGBoost — Métricas de la mejor configuración en validación cruzada"
)
ax.set_xlabel("Métrica")
ax.set_ylabel("Puntuación promedio")
ax.set_ylim(0, 1)

ax.bar_label(
    ax.containers[0],
    fmt="%.4f",
    padding=3
)

ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

---

## Entrenamiento del modelo XGBoost definitivo

Después de completar la búsqueda de hiperparámetros, se recupera el mejor estimador mediante `best_estimator_`.

Gracias a `refit=True`, este modelo ya fue ajustado automáticamente con la totalidad del conjunto de entrenamiento utilizando la mejor combinación de hiperparámetros encontrada.

Por tanto, no es necesario volver a ejecutar `fit()` sobre el mismo conjunto de entrenamiento.

El modelo definitivo se utilizará para generar predicciones sobre el conjunto de prueba y analizar su capacidad de generalización.

In [ ]:
xgb_model = random_search.best_estimator_

print("Modelo XGBoost seleccionado:")
print(xgb_model)

---

## Evaluación final del modelo sobre el conjunto de prueba

Se generan predicciones para las observaciones de `X_test`.

Las categorías predichas se comparan con los valores reales de `y_test` mediante las mismas métricas utilizadas durante la búsqueda de hiperparámetros.

Esta evaluación permite medir la capacidad de generalización del modelo sobre datos que no participaron en la selección de hiperparámetros ni en el entrenamiento final.

También se genera un reporte de clasificación que presenta, para cada categoría de popularidad, su precisión, recall, F1-score y cantidad de observaciones.

La comparación entre los resultados de validación cruzada y los del conjunto de prueba permite identificar posibles diferencias de rendimiento. Estas diferencias deben interpretarse considerando la distribución de las clases y el tamaño de los conjuntos evaluados.

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

print("Resultados de XGBoost en el conjunto de prueba:")

print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")

print(
    f"Balanced Accuracy: "
    f"{balanced_accuracy_score(y_test, y_pred_xgb):.4f}"
)

print(
    f"Precision: "
    f"{precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0):.4f}"
)

print(
    f"Recall: "
    f"{recall_score(y_test, y_pred_xgb, average='weighted', zero_division=0):.4f}"
)

print(
    f"F1-Weighted: "
    f"{f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0):.4f}"
)

print(
    f"F1-Macro: "
    f"{f1_score(y_test, y_pred_xgb, average='macro', zero_division=0):.4f}"
)

print("\nReporte de clasificación:")

print(
    classification_report(
        y_test,
        y_pred_xgb,
        zero_division=0
    )
)

---

## Análisis de las matrices de confusión

Se construyen dos matrices de confusión para examinar con mayor detalle los aciertos y errores del modelo.

**Matriz de confusión en cantidades**

Muestra el número de observaciones correspondientes a cada combinación de categoría real y categoría predicha. La diagonal principal representa las predicciones correctas, mientras que los valores fuera de ella representan errores de clasificación.

**Matriz de confusión normalizada**

Presenta los resultados como proporciones respecto del total de observaciones de cada categoría real. Esta normalización permite comparar el comportamiento del modelo entre clases de diferentes tamaños.

En ambas matrices:

- Las filas representan las categorías reales.
- Las columnas representan las categorías predichas.
- La diagonal principal permite identificar el porcentaje o la cantidad de aciertos por categoría.
- Los valores fuera de la diagonal muestran qué categorías tiende a confundir el modelo.

Este análisis permite determinar si XGBoost distingue adecuadamente las categorías de popularidad o si presenta dificultades para diferenciar categorías cercanas.

In [ ]:
etiquetas = xgb_model.classes_

matriz = confusion_matrix(
    y_test,
    y_pred_xgb,
    labels=etiquetas
)

matriz_normalizada = confusion_matrix(
    y_test,
    y_pred_xgb,
    labels=etiquetas,
    normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=etiquetas,
    yticklabels=etiquetas,
    ax=axes[0]
)

axes[0].set_title("Matriz de confusión - Cantidades")
axes[0].set_xlabel("Categoría predicha")
axes[0].set_ylabel("Categoría real")

sns.heatmap(
    matriz_normalizada,
    annot=True,
    fmt=".1%",
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=etiquetas,
    yticklabels=etiquetas,
    ax=axes[1]
)

axes[1].set_title("Matriz de confusión - Porcentajes")
axes[1].set_xlabel("Categoría predicha")
axes[1].set_ylabel("Categoría real")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)

plt.suptitle(
    "Evaluación de XGBoost",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

---

## Importancia de las variables

Se examina la importancia de las características utilizadas por XGBoost para construir sus árboles de decisión.

La importancia permite identificar qué variables contribuyen en mayor medida a las decisiones del modelo, de acuerdo con el criterio de importancia proporcionado por XGBoost.

Se ordenan las características de mayor a menor importancia y se presentan las cinco primeras, junto con una visualización de las quince variables más relevantes.

**Consideración:** la importancia de variables describe la contribución dentro del modelo entrenado, pero no implica una relación causal entre una característica musical y la popularidad de una canción.

In [ ]:
importancias = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
})

importancias = importancias.sort_values(
    by="importance",
    ascending=False
)

print("Top 5 características más importantes:")

display(importancias.head(5))

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=importancias.head(15),
    x="importance",
    y="feature"
)

plt.title(
    "Top 15 características más importantes - XGBoost",
    fontsize=14,
    fontweight="bold"
)
plt.xlabel("Importancia")
plt.ylabel("Característica")
plt.tight_layout()
plt.show()